<a href="https://colab.research.google.com/github/Ahmadlazz/Data_Acqusition_Management_UTS/blob/main/UTS_Data_Acquisition.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

1




In [20]:
import pandas as pd
import requests
from bs4 import BeautifulSoup

import os
import time
import hashlib
import re

from datetime import datetime
from urllib.robotparser import RobotFileParser
from urllib.parse import urlparse

2

In [21]:
# Membuat folder penyimpanan
os.makedirs("data/raw/download", exist_ok=True)
os.makedirs("data/raw/scraping", exist_ok=True)
os.makedirs("data/process", exist_ok=True)
os.makedirs("docs", exist_ok=True)

# Masukkan email kampus sendiri
EMAIL_KAMPUS = "GANTI_DENGAN_EMAIL_KAMPUS"

# Identitas request
HEADERS = {
    "User-Agent": f"UTS-Data-Acquisition/1.0 ({EMAIL_KAMPUS})"
}

# Pengaturan request
TIMEOUT = 20
DELAY = 2

print("Persiapan proyek berhasil")

Persiapan proyek berhasil


## Akuisisi Dataset Terbuka

Dataset diperoleh dari portal data pemerintah
yang berkaitan dengan sektor pariwisata Bali.

In [22]:
# Melihat file yang tersedia
os.listdir("data/raw/download")

['Jumlah Kunjungan Wisatawan Mancanegara per bulan ke Indonesia Menurut Pintu Masuk, 2008 - sekarang, 2022.csv',
 'Jumlah Kunjungan Wisatawan Mancanegara per bulan ke Indonesia Menurut Pintu Masuk, 2008 - sekarang, 2023.csv',
 'Jumlah Kunjungan Wisatawan Mancanegara per bulan ke Indonesia Menurut Pintu Masuk, 2008 - sekarang, 2025.csv',
 '.ipynb_checkpoints',
 'Jumlah Kunjungan Wisatawan Mancanegara per bulan ke Indonesia Menurut Pintu Masuk, 2008 - sekarang, 2024.csv']

3

In [23]:
# Mengambil nama file CSV dari folder
file_dataset = [
    file for file in os.listdir("data/raw/download")
    if file.endswith(".csv")
]

# Memastikan file tersedia
if not file_dataset:
    raise FileNotFoundError("File CSV belum ditemukan")

# Membaca dataset
lokasi_file = "data/raw/download/" + file_dataset[0]

df_download = pd.read_csv(lokasi_file)

# Menampilkan lima data pertama
df_download.head()

,Pintu Masuk,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,Unnamed: 10,Unnamed: 11,Unnamed: 12,Unnamed: 13
0,NaN,Jumlah Kunjungan Wisatawan Mancanegara per bul...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,2022,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,Januari,Februari,Maret,April,Mei,Juni,Juli,Agustus,September,Oktober,November,Desember,Tahunan
3,A. Pintu Udara,14555,17834,39062,97401,183830,280482,387184,422500,424198,447074,429979,551927,3296026
4,Ngurah Rai,0,1293,14617,58315,115553,181545,246442,276627,291115,305152,287025,376361,2154045


4

In [24]:
# Jumlah baris dan kolom
print("Jumlah baris:", df_download.shape[0])
print("Jumlah kolom:", df_download.shape[1])

# Informasi tipe data
df_download.info()

# Statistik data
display(df_download.describe(include="all"))

# Melihat nama kolom
print(df_download.columns.tolist())

Jumlah baris: 38
Jumlah kolom: 14
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 38 entries, 0 to 37
Data columns (total 14 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   Pintu Masuk  35 non-null     object
 1   Unnamed: 1   38 non-null     object
 2   Unnamed: 2   36 non-null     object
 3   Unnamed: 3   36 non-null     object
 4   Unnamed: 4   36 non-null     object
 5   Unnamed: 5   36 non-null     object
 6   Unnamed: 6   36 non-null     object
 7   Unnamed: 7   36 non-null     object
 8   Unnamed: 8   36 non-null     object
 9   Unnamed: 9   36 non-null     object
 10  Unnamed: 10  36 non-null     object
 11  Unnamed: 11  36 non-null     object
 12  Unnamed: 12  36 non-null     object
 13  Unnamed: 13  36 non-null     object
dtypes: object(14)
memory usage: 4.3+ KB


,Pintu Masuk,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,Unnamed: 10,Unnamed: 11,Unnamed: 12,Unnamed: 13
count,35,38,36,36,36,36,36,36,36,36,36,36,36,36
unique,35,18,18,21,24,26,28,30,30,29,32,31,31,33
top,A. Pintu Udara,0,0,0,0,0,0,0,0,0,0,0,0,0
freq,1,19,17,14,10,9,7,5,5,6,3,4,4,2


['Pintu Masuk', 'Unnamed: 1', 'Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4', 'Unnamed: 5', 'Unnamed: 6', 'Unnamed: 7', 'Unnamed: 8', 'Unnamed: 9', 'Unnamed: 10', 'Unnamed: 11', 'Unnamed: 12', 'Unnamed: 13']


5

In [25]:
if len(df_download) >= 100:
    print("Dataset memenuhi minimal 100 baris")
else:
    print("Dataset belum memenuhi minimal 100 baris")

Dataset belum memenuhi minimal 100 baris


6

In [26]:
# Menghitung ukuran file dalam byte
ukuran_byte = os.path.getsize(lokasi_file)

# Mengubah byte menjadi MB
ukuran_mb = ukuran_byte / (1024 ** 2)

# Estimasi kebutuhan selama 12 bulan
estimasi_tahunan = ukuran_mb * 12

# Menambahkan cadangan 30 persen
estimasi_total = estimasi_tahunan * 1.30

print("Ukuran file:", round(ukuran_mb, 4), "MB")
print("Estimasi 1 tahun:", round(estimasi_tahunan, 4), "MB")
print("Dengan cadangan 30%:", round(estimasi_total, 4), "MB")

Ukuran file: 0.0026 MB
Estimasi 1 tahun: 0.031 MB
Dengan cadangan 30%: 0.0403 MB


## Pemeriksaan Etika Data

Pemeriksaan dilakukan terhadap:
1. Lisensi dataset.
2. Ketentuan penggunaan portal.
3. Data pribadi atau PII.
4. Quasi-identifier.
5. Kesesuaian penggunaan data dengan tujuan proyek.

Bukti pemeriksaan dicatat dalam
data_ethics_checklist.md.

7

In [27]:
# Menghitung data kosong
print(df_download.isnull().sum())

# Memeriksa nama kolom yang mungkin mengandung data pribadi
kata_pii = [
    "nama", "email", "telepon",
    "phone", "alamat", "nik",
    "identitas"
]

kolom_pii = [
    kolom for kolom in df_download.columns
    if any(kata in kolom.lower() for kata in kata_pii)
]

print("Kolom yang perlu diperiksa:", kolom_pii)

Pintu Masuk    3
Unnamed: 1     0
Unnamed: 2     2
Unnamed: 3     2
Unnamed: 4     2
Unnamed: 5     2
Unnamed: 6     2
Unnamed: 7     2
Unnamed: 8     2
Unnamed: 9     2
Unnamed: 10    2
Unnamed: 11    2
Unnamed: 12    2
Unnamed: 13    2
dtype: int64
Kolom yang perlu diperiksa: []


8

In [28]:
URL_TARGET = (
    "https://www.daybalitour.com/"
    "best-bali-day-tour-packages/"
)

URL_ROBOTS = "https://www.daybalitour.com/robots.txt"

# Mengambil robots.txt
response_robots = requests.get(
    URL_ROBOTS,
    headers=HEADERS,
    timeout=TIMEOUT
)

print("Status:", response_robots.status_code)
print(response_robots.text)

Status: 200
User-agent: *
Disallow: /wp-admin/
Allow: /wp-admin/admin-ajax.php
Disallow: /wp-content/uploads/wpforms/

Sitemap: https://www.daybalitour.com/sitemap.xml
Sitemap: https://www.daybalitour.com/sitemap.rss



9

10

In [29]:
robot_parser = RobotFileParser()
robot_parser.set_url(URL_ROBOTS)

robot_parser.read()

boleh_scraping = robot_parser.can_fetch(
    HEADERS["User-Agent"],
    URL_TARGET
)

print("Boleh menurut robots.txt:", boleh_scraping)

Boleh menurut robots.txt: True


In [30]:
if not boleh_scraping:
    raise PermissionError(
        "Halaman tidak diizinkan menurut robots.txt"
    )

# Mengambil halaman
response = requests.get(
    URL_TARGET,
    headers=HEADERS,
    timeout=TIMEOUT
)

# Memeriksa status respons
print("Status HTTP:", response.status_code)

response.raise_for_status()

# Memberikan jeda
time.sleep(DELAY)

print("Halaman berhasil diakses")

Status HTTP: 200
Halaman berhasil diakses


11

In [31]:
# Menyimpan HTML asli
file_html = "data/raw/scraping/day_bali_tour.html"

with open(file_html, "w", encoding="utf-8") as file:
    file.write(response.text)

print("HTML mentah berhasil disimpan")

HTML mentah berhasil disimpan


12

In [32]:
soup = BeautifulSoup(response.text, "html.parser")

# Menampilkan judul halaman
print(soup.title.get_text(strip=True))

# Menghitung jumlah heading
print("Jumlah heading:", len(soup.find_all(["h2", "h3", "h4"])))

Best Bali Day Tour Packages - Day Bali Tour
Jumlah heading: 22


13

In [33]:
hasil_scraping = []

# Mencari heading paket wisata
for heading in soup.find_all(["h2", "h3", "h4"]):

    nama_paket = heading.get_text(" ", strip=True)

    # Mencari elemen sekitar heading
    parent = heading.parent

    teks_parent = parent.get_text(
        " ",
        strip=True
    )

    # Mencari harga dalam teks
    harga = re.search(
        r"(?:IDR|Rp)\s?[\d.,]+",
        teks_parent,
        re.IGNORECASE
    )

    # Menyimpan hasil
    hasil_scraping.append({
        "nama_paket": nama_paket,
        "harga": harga.group(0) if harga else None,
        "informasi": teks_parent
    })

# Membuat DataFrame
df_scraping = pd.DataFrame(hasil_scraping)

display(df_scraping.head(10))

,nama_paket,harga,informasi
0,Bali Day Tours Highlights,None,Bali Day Tours Highlights
1,Bali Waterfall and Best Ubud Tour Packages,IDR 580.000,Bali Waterfall and Best Ubud Tour Packages Fro...
2,From IDR 580.000,IDR 580.000,From IDR 580.000 What’s Included: Private Hote...
3,Bali Best Nusa Penida Island Tour Packages,IDR 760.000,Bali Best Nusa Penida Island Tour Packages Fro...
4,From IDR 760.000,IDR 760.000,From IDR 760.000 What’s Included: Private Hote...
5,Bali Lempuyang Gate of Heaven Tour Package,IDR 745.000,Bali Lempuyang Gate of Heaven Tour Package Fro...
6,From IDR 745.000,IDR 745.000,From IDR 745.000 What’s Included: Private Hote...
7,Bali Ulun Danu and Tanah Lot Tour Package,IDR 790.000,Bali Ulun Danu and Tanah Lot Tour Package From...
8,From IDR 790.000,IDR 790.000,From IDR 790.000 What’s Included: Private Hote...
9,Bali Tanjung Benoa & Uluwatu Tour Package,IDR 870.000,Bali Tanjung Benoa & Uluwatu Tour Package From...


14

In [34]:
# Menghapus baris yang seluruh kolomnya kosong
df_scraping = df_scraping.dropna(how="all")

# Menghapus baris duplikat
df_scraping = df_scraping.drop_duplicates()

# Menghapus spasi berlebih pada teks
for kolom in df_scraping.select_dtypes(include="object"):
    df_scraping[kolom] = df_scraping[kolom].str.strip()

# Mengatur ulang nomor indeks
df_scraping = df_scraping.reset_index(drop=True)

display(df_scraping)

,nama_paket,harga,informasi
0,Bali Day Tours Highlights,None,Bali Day Tours Highlights
1,Bali Waterfall and Best Ubud Tour Packages,IDR 580.000,Bali Waterfall and Best Ubud Tour Packages Fro...
2,From IDR 580.000,IDR 580.000,From IDR 580.000 What’s Included: Private Hote...
3,Bali Best Nusa Penida Island Tour Packages,IDR 760.000,Bali Best Nusa Penida Island Tour Packages Fro...
4,From IDR 760.000,IDR 760.000,From IDR 760.000 What’s Included: Private Hote...
5,Bali Lempuyang Gate of Heaven Tour Package,IDR 745.000,Bali Lempuyang Gate of Heaven Tour Package Fro...
6,From IDR 745.000,IDR 745.000,From IDR 745.000 What’s Included: Private Hote...
7,Bali Ulun Danu and Tanah Lot Tour Package,IDR 790.000,Bali Ulun Danu and Tanah Lot Tour Package From...
8,From IDR 790.000,IDR 790.000,From IDR 790.000 What’s Included: Private Hote...
9,Bali Tanjung Benoa & Uluwatu Tour Package,IDR 870.000,Bali Tanjung Benoa & Uluwatu Tour Package From...


15

In [35]:
# Menyimpan hasil ke CSV
df_scraping.to_csv(
    "data/process/hasil_scraping.csv",
    index=False
)

# Menyimpan hasil ke JSON
df_scraping.to_json(
    "data/process/hasil_scraping.json",
    orient="records",
    indent=4,
    force_ascii=False
)

print("Hasil scraping berhasil disimpan")

Hasil scraping berhasil disimpan


In [36]:
print("Kolom dataset unduhan:")
print(df_download.columns.tolist())

print("\nKolom data scraping:")
print(df_scraping.columns.tolist())

Kolom dataset unduhan:
['Pintu Masuk', 'Unnamed: 1', 'Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4', 'Unnamed: 5', 'Unnamed: 6', 'Unnamed: 7', 'Unnamed: 8', 'Unnamed: 9', 'Unnamed: 10', 'Unnamed: 11', 'Unnamed: 12', 'Unnamed: 13']

Kolom data scraping:
['nama_paket', 'harga', 'informasi']


16

In [37]:
def hitung_md5(lokasi_file):

    md5 = hashlib.md5()

    with open(lokasi_file, "rb") as file:

        for bagian in iter(
            lambda: file.read(4096),
            b""
        ):
            md5.update(bagian)

    return md5.hexdigest()

17

In [38]:
print(
    hitung_md5(
        "data/raw/scraping/day_bali_tour.html"
    )
)

a2d4750647a51c5252cb8b8bf0d28996


In [48]:
# Membuat nama kolom yang benar

nama_kolom = [
    "Pintu Masuk",
    "Januari",
    "Februari",
    "Maret",
    "April",
    "Mei",
    "Juni",
    "Juli",
    "Agustus",
    "September",
    "Oktober",
    "November",
    "Desember",
    "Tahunan"
]

# Mengambil data mulai baris ke-4
df_download_bersih = df_download.iloc[3:].copy()

# Mengganti nama kolom
df_download_bersih.columns = nama_kolom

# Menghapus baris yang tidak memiliki nama pintu masuk
df_download_bersih = df_download_bersih.dropna(
    subset=["Pintu Masuk"]
)

# Merapikan nama pintu masuk
df_download_bersih["Pintu Masuk"] = (
    df_download_bersih["Pintu Masuk"].astype(str).str.strip()
)

# Mengubah data jumlah kunjungan menjadi angka
kolom_angka = nama_kolom[1:]

for kolom in kolom_angka:
    df_download_bersih[kolom] = pd.to_numeric(
        df_download_bersih[kolom],
        errors="coerce"
    )

# Menampilkan hasil
display(df_download_bersih.head(10))

print("Jumlah baris:", len(df_download_bersih))
print("Jumlah kolom:", len(df_download_bersih.columns))

,Pintu Masuk,Januari,Februari,Maret,April,Mei,Juni,Juli,Agustus,September,Oktober,November,Desember,Tahunan
3,A. Pintu Udara,14555.0,17834.0,39062.0,97401.0,183830.0,280482.0,387184.0,422500.0,424198.0,447074.0,429979.0,551927.0,3296026.0
4,Ngurah Rai,0.0,1293.0,14617.0,58315.0,115553.0,181545.0,246442.0,276627.0,291115.0,305152.0,287025.0,376361.0,2154045.0
5,Soekarno-Hatta,14073.0,15406.0,23320.0,36061.0,57844.0,85587.0,119179.0,121427.0,111033.0,112527.0,113729.0,124475.0,934661.0
6,Juanda,0.0,0.0,57.0,1468.0,4598.0,4722.0,6087.0,8386.0,9152.0,9415.0,9644.0,14264.0,67793.0
7,Kualanamu,0.0,0.0,11.0,494.0,3861.0,5165.0,7518.0,10287.0,5895.0,11454.0,10615.0,19198.0,74498.0
8,Husein Sastranegara,0.0,0.0,0.0,0.0,0.0,49.0,21.0,15.0,0.0,0.0,0.0,0.0,85.0
9,Adi Sucipto,0.0,0.0,0.0,35.0,260.0,369.0,809.0,640.0,974.0,1933.0,1947.0,5169.0,12136.0
10,Bandara Int. Lombok,0.0,488.0,12.0,0.0,487.0,1011.0,1658.0,2054.0,2251.0,2226.0,1921.0,2914.0,15022.0
11,Sam Ratulangi,466.0,647.0,1045.0,851.0,843.0,1472.0,1684.0,1758.0,2178.0,1581.0,1721.0,1142.0,15388.0
12,Minangkabau,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,371.0,709.0,3062.0,4142.0


Jumlah baris: 35
Jumlah kolom: 14


In [45]:
# Contoh setelah df_gabungan berhasil dibuat
df_gabungan.to_csv(
    "data/process/hasil_gabungan.csv",
    index=False
)

print("Hasil penggabungan berhasil disimpan")

NameError: name 'df_gabungan' is not defined

In [40]:
print("Jumlah baris:", len(df_gabungan))

print("Jumlah duplikasi:")
print(df_gabungan.duplicated().sum())

display(df_gabungan.head())

NameError: name 'df_gabungan' is not defined

18

In [41]:
# Membuat daftar informasi sumber
daftar_provenance = []

file_sumber = [
    "data/raw/scraping/day_bali_tour.html"
]

for file in file_sumber:

    ukuran = os.path.getsize(file)

    informasi = {
        "nama_file": os.path.basename(file),
        "ukuran_byte": ukuran,
        "md5": hitung_md5(file),
        "tanggal_akses": datetime.now().isoformat()
    }

    daftar_provenance.append(informasi)

# Membuat tabel provenance
df_provenance = pd.DataFrame(daftar_provenance)

display(df_provenance)

,nama_file,ukuran_byte,md5,tanggal_akses
0,day_bali_tour.html,391767,a2d4750647a51c5252cb8b8bf0d28996,2026-11-24T14:46:34.722206


10

In [ ]:
df_provenance.to_csv(
    "docs/provenance.csv",
    index=False
)

print("Provenance berhasil disimpan")

In [ ]:
for folder, subfolder, files in os.walk("."):

    if ".ipynb_checkpoints" in folder:
        continue

    print("\nFolder:", folder)

    for nama_file in files:
        print("-", nama_file)

In [19]:
import os

folder = "data/raw/download"

print("Daftar file dalam folder download:")

for file in os.listdir(folder):
    print("-", file)

Daftar file dalam folder download:
- Jumlah Kunjungan Wisatawan Mancanegara per bulan ke Indonesia Menurut Pintu Masuk, 2008 - sekarang, 2022.csv
- Jumlah Kunjungan Wisatawan Mancanegara per bulan ke Indonesia Menurut Pintu Masuk, 2008 - sekarang, 2023.csv
- Jumlah Kunjungan Wisatawan Mancanegara per bulan ke Indonesia Menurut Pintu Masuk, 2008 - sekarang, 2025.csv
- .ipynb_checkpoints
- Jumlah Kunjungan Wisatawan Mancanegara per bulan ke Indonesia Menurut Pintu Masuk, 2008 - sekarang, 2024.csv


In [42]:
URL_SCRAPING = "https://www.daybalitour.com/best-bali-day-tour-packages/"

HEADERS = {
    "User-Agent": "Mahasiswa Informatika - email_kampus_kamu"
}

TIMEOUT = 20
DELAY = 2

In [43]:
import requests
from bs4 import BeautifulSoup
import time

time.sleep(DELAY)

response = requests.get(
    URL_SCRAPING,
    headers=HEADERS,
    timeout=TIMEOUT
)

print("Status Code:", response.status_code)

response.raise_for_status()

soup = BeautifulSoup(response.text, "html.parser")

print("Judul halaman:", soup.title.get_text(strip=True))

for heading in soup.find_all(["h2", "h3", "h4"]):
    print(heading.name, ":", heading.get_text(" ", strip=True))

Status Code: 200
Judul halaman: Best Bali Day Tour Packages - Day Bali Tour
h2 : Bali Day Tours Highlights
h3 : Bali Waterfall and Best Ubud Tour Packages
h4 : From IDR 580.000
h3 : Bali Best Nusa Penida Island Tour Packages
h4 : From IDR 760.000
h3 : Bali Lempuyang Gate of Heaven Tour Package
h4 : From IDR 745.000
h3 : Bali Ulun Danu and Tanah Lot Tour Package
h4 : From IDR 790.000
h3 : Bali Tanjung Benoa & Uluwatu Tour Package
h4 : From IDR 870.000
h3 : Bali Kintamani Volcano View Tour Package
h4 : From IDR 715.000
h4 : Special Offer
h4 : Bali Activities
h2 : Bali Driver (Tour with Your Itinerary)
h3 : Bali Driver with Toyota Avanza
h4 : From 375.000/Car
h3 : Bali Driver with Toyota Innova
h4 : IDR 1.125.000/Car
h3 : Bali Driver with Toyota Hiace
h4 : IDR 1.275.000/Car


In [44]:
# Menampilkan seluruh output tanpa terpotong

from IPython.display import display

for heading in soup.find_all(["h2", "h3", "h4"]):
    print(heading.name, ":", heading.get_text(" ", strip=True))

h2 : Bali Day Tours Highlights
h3 : Bali Waterfall and Best Ubud Tour Packages
h4 : From IDR 580.000
h3 : Bali Best Nusa Penida Island Tour Packages
h4 : From IDR 760.000
h3 : Bali Lempuyang Gate of Heaven Tour Package
h4 : From IDR 745.000
h3 : Bali Ulun Danu and Tanah Lot Tour Package
h4 : From IDR 790.000
h3 : Bali Tanjung Benoa & Uluwatu Tour Package
h4 : From IDR 870.000
h3 : Bali Kintamani Volcano View Tour Package
h4 : From IDR 715.000
h4 : Special Offer
h4 : Bali Activities
h2 : Bali Driver (Tour with Your Itinerary)
h3 : Bali Driver with Toyota Avanza
h4 : From 375.000/Car
h3 : Bali Driver with Toyota Innova
h4 : IDR 1.125.000/Car
h3 : Bali Driver with Toyota Hiace
h4 : IDR 1.275.000/Car


In [46]:
print("Kolom dataset download:")
print(df_download.columns.tolist())

print("\nKolom dataset scraping:")
print(df_scraping.columns.tolist())

Kolom dataset download:
['Pintu Masuk', 'Unnamed: 1', 'Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4', 'Unnamed: 5', 'Unnamed: 6', 'Unnamed: 7', 'Unnamed: 8', 'Unnamed: 9', 'Unnamed: 10', 'Unnamed: 11', 'Unnamed: 12', 'Unnamed: 13']

Kolom dataset scraping:
['nama_paket', 'harga', 'informasi']


In [47]:
print("BENTUK DATASET DOWNLOAD:")
print(df_download.shape)

print("\n10 BARIS PERTAMA DATASET DOWNLOAD:")
display(df_download.head(10))

print("\nINFORMASI DATASET DOWNLOAD:")
df_download.info()

BENTUK DATASET DOWNLOAD:
(38, 14)

10 BARIS PERTAMA DATASET DOWNLOAD:


,Pintu Masuk,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,Unnamed: 10,Unnamed: 11,Unnamed: 12,Unnamed: 13
0,NaN,Jumlah Kunjungan Wisatawan Mancanegara per bul...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,2022,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,Januari,Februari,Maret,April,Mei,Juni,Juli,Agustus,September,Oktober,November,Desember,Tahunan
3,A. Pintu Udara,14555,17834,39062,97401,183830,280482,387184,422500,424198,447074,429979,551927,3296026
4,Ngurah Rai,0,1293,14617,58315,115553,181545,246442,276627,291115,305152,287025,376361,2154045
5,Soekarno-Hatta,14073,15406,23320,36061,57844,85587,119179,121427,111033,112527,113729,124475,934661
6,Juanda,0,0,57,1468,4598,4722,6087,8386,9152,9415,9644,14264,67793
7,Kualanamu,0,0,11,494,3861,5165,7518,10287,5895,11454,10615,19198,74498
8,Husein Sastranegara,0,0,0,0,0,49,21,15,0,0,0,0,85
9,Adi Sucipto,0,0,0,35,260,369,809,640,974,1933,1947,5169,12136



INFORMASI DATASET DOWNLOAD:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 38 entries, 0 to 37
Data columns (total 14 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   Pintu Masuk  35 non-null     object
 1   Unnamed: 1   38 non-null     object
 2   Unnamed: 2   36 non-null     object
 3   Unnamed: 3   36 non-null     object
 4   Unnamed: 4   36 non-null     object
 5   Unnamed: 5   36 non-null     object
 6   Unnamed: 6   36 non-null     object
 7   Unnamed: 7   36 non-null     object
 8   Unnamed: 8   36 non-null     object
 9   Unnamed: 9   36 non-null     object
 10  Unnamed: 10  36 non-null     object
 11  Unnamed: 11  36 non-null     object
 12  Unnamed: 12  36 non-null     object
 13  Unnamed: 13  36 non-null     object
dtypes: object(14)
memory usage: 4.3+ KB


In [49]:
print("Jumlah baris:", len(df_download_bersih))
print("Jumlah kolom:", len(df_download_bersih.columns))

print("\nNama kolom:")
print(df_download_bersih.columns.tolist())

print("\nData setelah dibersihkan:")
display(df_download_bersih.head(10))

print("\nTipe data:")
df_download_bersih.info()

Jumlah baris: 35
Jumlah kolom: 14

Nama kolom:
['Pintu Masuk', 'Januari', 'Februari', 'Maret', 'April', 'Mei', 'Juni', 'Juli', 'Agustus', 'September', 'Oktober', 'November', 'Desember', 'Tahunan']

Data setelah dibersihkan:


,Pintu Masuk,Januari,Februari,Maret,April,Mei,Juni,Juli,Agustus,September,Oktober,November,Desember,Tahunan
3,A. Pintu Udara,14555.0,17834.0,39062.0,97401.0,183830.0,280482.0,387184.0,422500.0,424198.0,447074.0,429979.0,551927.0,3296026.0
4,Ngurah Rai,0.0,1293.0,14617.0,58315.0,115553.0,181545.0,246442.0,276627.0,291115.0,305152.0,287025.0,376361.0,2154045.0
5,Soekarno-Hatta,14073.0,15406.0,23320.0,36061.0,57844.0,85587.0,119179.0,121427.0,111033.0,112527.0,113729.0,124475.0,934661.0
6,Juanda,0.0,0.0,57.0,1468.0,4598.0,4722.0,6087.0,8386.0,9152.0,9415.0,9644.0,14264.0,67793.0
7,Kualanamu,0.0,0.0,11.0,494.0,3861.0,5165.0,7518.0,10287.0,5895.0,11454.0,10615.0,19198.0,74498.0
8,Husein Sastranegara,0.0,0.0,0.0,0.0,0.0,49.0,21.0,15.0,0.0,0.0,0.0,0.0,85.0
9,Adi Sucipto,0.0,0.0,0.0,35.0,260.0,369.0,809.0,640.0,974.0,1933.0,1947.0,5169.0,12136.0
10,Bandara Int. Lombok,0.0,488.0,12.0,0.0,487.0,1011.0,1658.0,2054.0,2251.0,2226.0,1921.0,2914.0,15022.0
11,Sam Ratulangi,466.0,647.0,1045.0,851.0,843.0,1472.0,1684.0,1758.0,2178.0,1581.0,1721.0,1142.0,15388.0
12,Minangkabau,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,371.0,709.0,3062.0,4142.0



Tipe data:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 35 entries, 3 to 37
Data columns (total 14 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Pintu Masuk  35 non-null     object 
 1   Januari      33 non-null     float64
 2   Februari     33 non-null     float64
 3   Maret        33 non-null     float64
 4   April        33 non-null     float64
 5   Mei          33 non-null     float64
 6   Juni         33 non-null     float64
 7   Juli         33 non-null     float64
 8   Agustus      33 non-null     float64
 9   September    33 non-null     float64
 10  Oktober      33 non-null     float64
 11  November     33 non-null     float64
 12  Desember     33 non-null     float64
 13  Tahunan      33 non-null     float64
dtypes: float64(13), object(1)
memory usage: 4.0+ KB


In [51]:
import os

folder_data = "data/raw/download"

print("Daftar file CSV:\n")

for file in os.listdir(folder_data):
    if file.lower().endswith(".csv"):
        print(file)

Daftar file CSV:

Jumlah Kunjungan Wisatawan Mancanegara per bulan ke Indonesia Menurut Pintu Masuk, 2008 - sekarang, 2022.csv
Jumlah Kunjungan Wisatawan Mancanegara per bulan ke Indonesia Menurut Pintu Masuk, 2008 - sekarang, 2023.csv
Jumlah Kunjungan Wisatawan Mancanegara per bulan ke Indonesia Menurut Pintu Masuk, 2008 - sekarang, 2025.csv
Jumlah Kunjungan Wisatawan Mancanegara per bulan ke Indonesia Menurut Pintu Masuk, 2008 - sekarang, 2024.csv


In [52]:
import pandas as pd
import os

folder_data = "data/raw/download"

file_csv = [
    file for file in os.listdir(folder_data)
    if file.lower().endswith(".csv")
]

print("Jumlah dataset:", len(file_csv))

for file in sorted(file_csv):
    print("-", file)

Jumlah dataset: 4
- Jumlah Kunjungan Wisatawan Mancanegara per bulan ke Indonesia Menurut Pintu Masuk, 2008 - sekarang, 2022.csv
- Jumlah Kunjungan Wisatawan Mancanegara per bulan ke Indonesia Menurut Pintu Masuk, 2008 - sekarang, 2023.csv
- Jumlah Kunjungan Wisatawan Mancanegara per bulan ke Indonesia Menurut Pintu Masuk, 2008 - sekarang, 2024.csv
- Jumlah Kunjungan Wisatawan Mancanegara per bulan ke Indonesia Menurut Pintu Masuk, 2008 - sekarang, 2025.csv


In [53]:
# Membaca seluruh dataset berdasarkan tahun

dataset_tahunan = {}

for tahun in [2022, 2023, 2024, 2025]:

    nama_file = (
        f"Jumlah Kunjungan Wisatawan Mancanegara per bulan "
        f"ke Indonesia Menurut Pintu Masuk, 2008 - sekarang, {tahun}.csv"
    )

    lokasi_file = os.path.join(folder_data, nama_file)

    dataset_tahunan[tahun] = pd.read_csv(lokasi_file)

    print(f"Dataset {tahun} berhasil dibaca")

Dataset 2022 berhasil dibaca
Dataset 2023 berhasil dibaca
Dataset 2024 berhasil dibaca
Dataset 2025 berhasil dibaca


In [54]:
for tahun, data in dataset_tahunan.items():

    print(f"\n===== DATA TAHUN {tahun} =====")
    print("Jumlah baris:", len(data))
    print("Jumlah kolom:", len(data.columns))

    display(data.head(5))


===== DATA TAHUN 2022 =====
Jumlah baris: 38
Jumlah kolom: 14


,Pintu Masuk,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,Unnamed: 10,Unnamed: 11,Unnamed: 12,Unnamed: 13
0,NaN,Jumlah Kunjungan Wisatawan Mancanegara per bul...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,2022,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,Januari,Februari,Maret,April,Mei,Juni,Juli,Agustus,September,Oktober,November,Desember,Tahunan
3,A. Pintu Udara,14555,17834,39062,97401,183830,280482,387184,422500,424198,447074,429979,551927,3296026
4,Ngurah Rai,0,1293,14617,58315,115553,181545,246442,276627,291115,305152,287025,376361,2154045



===== DATA TAHUN 2023 =====
Jumlah baris: 38
Jumlah kolom: 14


,Pintu Masuk,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,Unnamed: 10,Unnamed: 11,Unnamed: 12,Unnamed: 13
0,NaN,Jumlah Kunjungan Wisatawan Mancanegara per bul...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,2023,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,Januari,Februari,Maret,April,Mei,Juni,Juli,Agustus,September,Oktober,November,Desember,Tahunan
3,A. Pintu Udara,484497,487449,554665,560434,659065,707476,819496,826446,777537,707584,656709,731293,7972651
4,Ngurah Rai,329909,317005,366956,410281,439454,478127,541272,522063,508297,458845,397522,478382,5248113



===== DATA TAHUN 2024 =====
Jumlah baris: 38
Jumlah kolom: 14


,Pintu Masuk,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,Unnamed: 10,Unnamed: 11,Unnamed: 12,Unnamed: 13
0,NaN,Jumlah Kunjungan Wisatawan Mancanegara per bul...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,2024,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,Januari,Februari,Maret,April,Mei,Juni,Juli,Agustus,September,Oktober,November,Desember,Tahunan
3,A. Pintu Udara,631961,735072,682439,750085,835722,821654,994583,987547,941903,859747,756972,832865,9830550
4,Ngurah Rai,414937,454398,463804,502870,544492,520792,625569,613540,591848,554892,472714,548685,6308541



===== DATA TAHUN 2025 =====
Jumlah baris: 38
Jumlah kolom: 14


,Pintu Masuk,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,Unnamed: 10,Unnamed: 11,Unnamed: 12,Unnamed: 13
0,NaN,Jumlah Kunjungan Wisatawan Mancanegara per bul...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,2025,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,Januari,Februari,Maret,April,Mei,Juni,Juli,Agustus,September,Oktober,November,Desember,Tahunan
3,A. Pintu Udara,801614,718609,669479,854756,941092,978929,1101619,1108133,996518,954216,805662,888664,10819291
4,Ngurah Rai,526831,444411,464697,587315,602055,637744,697068,682839,634482,594846,476507,558790,6907585


In [55]:
# Daftar nama kolom dataset

nama_kolom = [
    "Pintu Masuk",
    "Januari",
    "Februari",
    "Maret",
    "April",
    "Mei",
    "Juni",
    "Juli",
    "Agustus",
    "September",
    "Oktober",
    "November",
    "Desember",
    "Tahunan"
]


# Fungsi untuk membersihkan dataset

def bersihkan_dataset(data, tahun):

    # Mengambil data mulai baris ke-4
    data_bersih = data.iloc[3:].copy()

    # Memberikan nama kolom
    data_bersih.columns = nama_kolom

    # Menghapus baris tanpa nama pintu masuk
    data_bersih = data_bersih.dropna(
        subset=["Pintu Masuk"]
    )

    # Merapikan nama pintu masuk
    data_bersih["Pintu Masuk"] = (
        data_bersih["Pintu Masuk"]
        .astype(str)
        .str.strip()
    )

    # Mengubah kolom jumlah kunjungan menjadi angka
    kolom_angka = nama_kolom[1:]

    for kolom in kolom_angka:
        data_bersih[kolom] = pd.to_numeric(
            data_bersih[kolom],
            errors="coerce"
        )

    # Menambahkan identitas tahun
    data_bersih["Tahun"] = tahun

    return data_bersih


# Membersihkan empat dataset

dataset_bersih = {}

for tahun, data in dataset_tahunan.items():

    dataset_bersih[tahun] = bersihkan_dataset(
        data,
        tahun
    )

    print(f"Dataset {tahun} berhasil dibersihkan!")

Dataset 2022 berhasil dibersihkan!
Dataset 2023 berhasil dibersihkan!
Dataset 2024 berhasil dibersihkan!
Dataset 2025 berhasil dibersihkan!


In [56]:
for tahun, data in dataset_bersih.items():

    print(f"\n===== DATA TAHUN {tahun} =====")

    print("Jumlah baris:", len(data))
    print("Jumlah kolom:", len(data.columns))

    display(data.head(5))


===== DATA TAHUN 2022 =====
Jumlah baris: 35
Jumlah kolom: 15


,Pintu Masuk,Januari,Februari,Maret,April,Mei,Juni,Juli,Agustus,September,Oktober,November,Desember,Tahunan,Tahun
3,A. Pintu Udara,14555.0,17834.0,39062.0,97401.0,183830.0,280482.0,387184.0,422500.0,424198.0,447074.0,429979.0,551927.0,3296026.0,2022
4,Ngurah Rai,0.0,1293.0,14617.0,58315.0,115553.0,181545.0,246442.0,276627.0,291115.0,305152.0,287025.0,376361.0,2154045.0,2022
5,Soekarno-Hatta,14073.0,15406.0,23320.0,36061.0,57844.0,85587.0,119179.0,121427.0,111033.0,112527.0,113729.0,124475.0,934661.0,2022
6,Juanda,0.0,0.0,57.0,1468.0,4598.0,4722.0,6087.0,8386.0,9152.0,9415.0,9644.0,14264.0,67793.0,2022
7,Kualanamu,0.0,0.0,11.0,494.0,3861.0,5165.0,7518.0,10287.0,5895.0,11454.0,10615.0,19198.0,74498.0,2022



===== DATA TAHUN 2023 =====
Jumlah baris: 35
Jumlah kolom: 15


,Pintu Masuk,Januari,Februari,Maret,April,Mei,Juni,Juli,Agustus,September,Oktober,November,Desember,Tahunan,Tahun
3,A. Pintu Udara,484497.0,487449.0,554665.0,560434.0,659065.0,707476.0,819496.0,826446.0,777537.0,707584.0,656709.0,731293.0,7972651.0,2023
4,Ngurah Rai,329909.0,317005.0,366956.0,410281.0,439454.0,478127.0,541272.0,522063.0,508297.0,458845.0,397522.0,478382.0,5248113.0,2023
5,Soekarno-Hatta,115101.0,124803.0,140859.0,99289.0,158550.0,161114.0,204000.0,218512.0,190058.0,181627.0,186695.0,172397.0,1953005.0,2023
6,Juanda,10777.0,11882.0,10606.0,12361.0,15734.0,18025.0,20303.0,25548.0,25593.0,21996.0,22389.0,23244.0,218458.0,2023
7,Kualanamu,14146.0,15072.0,16106.0,15497.0,16262.0,16527.0,17478.0,20843.0,15783.0,14161.0,16363.0,18237.0,196475.0,2023



===== DATA TAHUN 2024 =====
Jumlah baris: 35
Jumlah kolom: 15


,Pintu Masuk,Januari,Februari,Maret,April,Mei,Juni,Juli,Agustus,September,Oktober,November,Desember,Tahunan,Tahun
3,A. Pintu Udara,631961.0,735072.0,682439.0,750085.0,835722.0,821654.0,994583.0,987547.0,941903.0,859747.0,756972.0,832865,9830550,2024
4,Ngurah Rai,414937.0,454398.0,463804.0,502870.0,544492.0,520792.0,625569.0,613540.0,591848.0,554892.0,472714.0,548685,6308541,2024
5,Soekarno-Hatta,157979.0,187641.0,160917.0,157409.0,201574.0,225985.0,271680.0,272849.0,250852.0,218170.0,212178.0,207019,2524253,2024
6,Juanda,17196.0,28026.0,18543.0,29043.0,31222.0,25015.0,34446.0,38587.0,31700.0,28182.0,20820.0,19265,322045,2024
7,Kualanamu,14751.0,24628.0,15068.0,25178.0,20469.0,16949.0,22693.0,21930.0,24956.0,20129.0,19145.0,21142,247038,2024



===== DATA TAHUN 2025 =====
Jumlah baris: 35
Jumlah kolom: 15


,Pintu Masuk,Januari,Februari,Maret,April,Mei,Juni,Juli,Agustus,September,Oktober,November,Desember,Tahunan,Tahun
3,A. Pintu Udara,801614,718609,669479,854756,941092,978929,1101619,1108133,996518,954216,805662,888664,10819291,2025
4,Ngurah Rai,526831,444411,464697,587315,602055,637744,697068,682839,634482,594846,476507,558790,6907585,2025
5,Soekarno-Hatta,193433,199306,146390,186523,233751,236013,290369,304970,244404,254968,240330,230381,2760838,2025
6,Juanda,22119,19205,15647,24800,33409,34092,34421,37406,32664,29534,21629,25019,329945,2025
7,Kualanamu,23011,22546,16881,19709,26183,25183,28306,29858,29107,24621,20826,23425,289656,2025


In [57]:
# Menggabungkan empat dataset tahunan

df_download_gabungan = pd.concat(
    dataset_bersih.values(),
    ignore_index=True
)

# Mengurutkan berdasarkan tahun dan pintu masuk
df_download_gabungan = df_download_gabungan.sort_values(
    by=["Tahun", "Pintu Masuk"]
).reset_index(drop=True)

print("Dataset berhasil digabungkan!")
print("Jumlah baris:", len(df_download_gabungan))
print("Jumlah kolom:", len(df_download_gabungan.columns))

display(df_download_gabungan.head(10))

Dataset berhasil digabungkan!
Jumlah baris: 140
Jumlah kolom: 15


,Pintu Masuk,Januari,Februari,Maret,April,Mei,Juni,Juli,Agustus,September,Oktober,November,Desember,Tahunan,Tahun
0,A. Pintu Udara,14555.0,17834.0,39062.0,97401.0,183830.0,280482.0,387184.0,422500.0,424198.0,447074.0,429979.0,551927.0,3296026.0,2022
1,Adi Sucipto,0.0,0.0,0.0,35.0,260.0,369.0,809.0,640.0,974.0,1933.0,1947.0,5169.0,12136.0,2022
2,Ahmad Yani,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2022
3,Aruk,0.0,0.0,0.0,218.0,622.0,969.0,1792.0,1149.0,1692.0,2116.0,1197.0,3020.0,12775.0,2022
4,Atambua,234.0,226.0,317.0,737.0,1203.0,1792.0,1149.0,3120.0,2672.0,2646.0,2511.0,4300.0,20907.0,2022
5,B. Pintu Laut,58442.0,51539.0,60357.0,69085.0,105719.0,134664.0,184193.0,164918.0,196668.0,207390.0,193894.0,299967.0,1726836.0,2022
6,Bandara Int. Lombok,0.0,488.0,12.0,0.0,487.0,1011.0,1658.0,2054.0,2251.0,2226.0,1921.0,2914.0,15022.0,2022
7,Batam,240.0,329.0,995.0,8136.0,16741.0,39649.0,56610.0,59750.0,77981.0,77752.0,80216.0,144521.0,562920.0,2022
8,C. Pintu Darat,63301.0,48136.0,59210.0,63590.0,65371.0,68737.0,73744.0,83091.0,79740.0,79764.0,80910.0,100575.0,866169.0,2022
9,Entikong,0.0,0.0,2.0,254.0,556.0,789.0,646.0,745.0,1360.0,1804.0,1771.0,3877.0,11804.0,2022


In [58]:
print("Jumlah data duplikat:")
print(df_download_gabungan.duplicated().sum())

print("\nJumlah data berdasarkan tahun:")
print(df_download_gabungan["Tahun"].value_counts().sort_index())

Jumlah data duplikat:
0

Jumlah data berdasarkan tahun:
Tahun
2022    35
2023    35
2024    35
2025    35
Name: count, dtype: int64


In [59]:
df_download_gabungan.to_csv(
    "data/process/hasil_download_gabungan.csv",
    index=False
)

print("File berhasil disimpan!")

File berhasil disimpan!


In [60]:
print("Daftar kategori Pintu Masuk:")

print(
    df_download_gabungan["Pintu Masuk"]
    .unique()
)

Daftar kategori Pintu Masuk:
['A. Pintu Udara' 'Adi Sucipto' 'Ahmad Yani' 'Aruk' 'Atambua'
 'B. Pintu Laut' 'Bandara Int. Lombok' 'Batam' 'C. Pintu Darat' 'Entikong'
 'Hasanuddin' 'Husein Sastranegara' 'Jayapura' 'Juanda' 'Jumlah (A+B+C)'
 'Kualanamu' 'Lainnya' 'Minangkabau' 'Nanga Badau' 'Ngurah Rai'
 'Pintu Darat Lainnya' 'Pintu Laut Lainnya' 'Pintu Udara Lainnya'
 'Sam Ratulangi' 'Soekarno-Hatta' 'Sultan Badaruddin II'
 'Sultan Iskandar Muda' 'Sultan Syarif Kasim II' 'Supadio'
 'Tanjung Balai Karimun' 'Tanjung Benoa' 'Tanjung Mas' 'Tanjung Pinang'
 'Tanjung Uban' 'Total']


In [61]:
# Daftar baris yang bukan data pintu masuk individual

baris_dihapus = [
    "A. Pintu Udara",
    "B. Pintu Laut",
    "C. Pintu Darat",
    "Jumlah (A+B+C)",
    "Total"
]

# Menghapus baris kategori dan rekapitulasi

df_download_detail = df_download_gabungan[
    ~df_download_gabungan["Pintu Masuk"].isin(baris_dihapus)
].copy()

# Memeriksa hasil

print("Jumlah baris sebelum:", len(df_download_gabungan))
print("Jumlah baris sesudah:", len(df_download_detail))

print("\nDaftar pintu masuk:")
print(df_download_detail["Pintu Masuk"].unique())

Jumlah baris sebelum: 140
Jumlah baris sesudah: 120

Daftar pintu masuk:
['Adi Sucipto' 'Ahmad Yani' 'Aruk' 'Atambua' 'Bandara Int. Lombok' 'Batam'
 'Entikong' 'Hasanuddin' 'Husein Sastranegara' 'Jayapura' 'Juanda'
 'Kualanamu' 'Lainnya' 'Minangkabau' 'Nanga Badau' 'Ngurah Rai'
 'Pintu Darat Lainnya' 'Pintu Laut Lainnya' 'Pintu Udara Lainnya'
 'Sam Ratulangi' 'Soekarno-Hatta' 'Sultan Badaruddin II'
 'Sultan Iskandar Muda' 'Sultan Syarif Kasim II' 'Supadio'
 'Tanjung Balai Karimun' 'Tanjung Benoa' 'Tanjung Mas' 'Tanjung Pinang'
 'Tanjung Uban']


In [62]:
# Memastikan folder penyimpanan tersedia
os.makedirs("data/process", exist_ok=True)

# Menyimpan dataset yang sudah dibersihkan
df_download_detail.to_csv(
    "data/process/hasil_download_bersih.csv",
    index=False
)

print("Dataset bersih berhasil disimpan!")
print("Jumlah baris:", len(df_download_detail))
print("Jumlah kolom:", len(df_download_detail.columns))

Dataset bersih berhasil disimpan!
Jumlah baris: 120
Jumlah kolom: 15


In [63]:
# Pemeriksaan akhir dataset bersih

print("JUMLAH BARIS:", len(df_download_detail))
print("JUMLAH KOLOM:", len(df_download_detail.columns))

print("\nJUMLAH DATA KOSONG:")
print(df_download_detail.isnull().sum())

print("\nJUMLAH DUPLIKASI:")
print(df_download_detail.duplicated().sum())

print("\nDATA BERDASARKAN TAHUN:")
print(df_download_detail["Tahun"].value_counts().sort_index())

print("\nCONTOH DATA:")
display(df_download_detail.head(10))

JUMLAH BARIS: 120
JUMLAH KOLOM: 15

JUMLAH DATA KOSONG:
Pintu Masuk    0
Januari        5
Februari       5
Maret          5
April          5
Mei            5
Juni           5
Juli           5
Agustus        5
September      5
Oktober        5
November       5
Desember       4
Tahunan        4
Tahun          0
dtype: int64

JUMLAH DUPLIKASI:
0

DATA BERDASARKAN TAHUN:
Tahun
2022    30
2023    30
2024    30
2025    30
Name: count, dtype: int64

CONTOH DATA:


,Pintu Masuk,Januari,Februari,Maret,April,Mei,Juni,Juli,Agustus,September,Oktober,November,Desember,Tahunan,Tahun
1,Adi Sucipto,0.0,0.0,0.0,35.0,260.0,369.0,809.0,640.0,974.0,1933.0,1947.0,5169.0,12136.0,2022
2,Ahmad Yani,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2022
3,Aruk,0.0,0.0,0.0,218.0,622.0,969.0,1792.0,1149.0,1692.0,2116.0,1197.0,3020.0,12775.0,2022
4,Atambua,234.0,226.0,317.0,737.0,1203.0,1792.0,1149.0,3120.0,2672.0,2646.0,2511.0,4300.0,20907.0,2022
6,Bandara Int. Lombok,0.0,488.0,12.0,0.0,487.0,1011.0,1658.0,2054.0,2251.0,2226.0,1921.0,2914.0,15022.0,2022
7,Batam,240.0,329.0,995.0,8136.0,16741.0,39649.0,56610.0,59750.0,77981.0,77752.0,80216.0,144521.0,562920.0,2022
9,Entikong,0.0,0.0,2.0,254.0,556.0,789.0,646.0,745.0,1360.0,1804.0,1771.0,3877.0,11804.0,2022
10,Hasanuddin,0.0,0.0,0.0,164.0,364.0,523.0,706.0,503.0,701.0,897.0,526.0,1530.0,5914.0,2022
11,Husein Sastranegara,0.0,0.0,0.0,0.0,0.0,49.0,21.0,15.0,0.0,0.0,0.0,0.0,85.0,2022
12,Jayapura,0.0,0.0,0.0,20.0,0.0,0.0,0.0,0.0,0.0,78.0,166.0,184.0,448.0,2022


In [64]:
# Melihat baris yang memiliki data kosong

data_kosong = df_download_detail[
    df_download_detail.isnull().any(axis=1)
]

print("Jumlah baris dengan data kosong:",
      len(data_kosong))

display(data_kosong)

Jumlah baris dengan data kosong: 5


,Pintu Masuk,Januari,Februari,Maret,April,Mei,Juni,Juli,Agustus,September,Oktober,November,Desember,Tahunan,Tahun
16,Lainnya,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2022
31,Tanjung Mas,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2022
51,Lainnya,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2023
66,Tanjung Mas,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2023
86,Lainnya,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,2024


In [65]:
# Daftar kolom bulan
kolom_bulan = [
    "Januari", "Februari", "Maret", "April",
    "Mei", "Juni", "Juli", "Agustus",
    "September", "Oktober", "November", "Desember"
]

# Menghapus baris yang seluruh data bulanannya kosong
df_download_final = df_download_detail.dropna(
    subset=kolom_bulan,
    how="all"
).copy()

# Memeriksa hasil
print("Jumlah baris sebelum:", len(df_download_detail))
print("Jumlah baris sesudah:", len(df_download_final))

print("\nData kosong tersisa:")
print(df_download_final.isnull().sum())

display(df_download_final.head(10))

Jumlah baris sebelum: 120
Jumlah baris sesudah: 116

Data kosong tersisa:
Pintu Masuk    0
Januari        1
Februari       1
Maret          1
April          1
Mei            1
Juni           1
Juli           1
Agustus        1
September      1
Oktober        1
November       1
Desember       0
Tahunan        0
Tahun          0
dtype: int64


,Pintu Masuk,Januari,Februari,Maret,April,Mei,Juni,Juli,Agustus,September,Oktober,November,Desember,Tahunan,Tahun
1,Adi Sucipto,0.0,0.0,0.0,35.0,260.0,369.0,809.0,640.0,974.0,1933.0,1947.0,5169.0,12136.0,2022
2,Ahmad Yani,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2022
3,Aruk,0.0,0.0,0.0,218.0,622.0,969.0,1792.0,1149.0,1692.0,2116.0,1197.0,3020.0,12775.0,2022
4,Atambua,234.0,226.0,317.0,737.0,1203.0,1792.0,1149.0,3120.0,2672.0,2646.0,2511.0,4300.0,20907.0,2022
6,Bandara Int. Lombok,0.0,488.0,12.0,0.0,487.0,1011.0,1658.0,2054.0,2251.0,2226.0,1921.0,2914.0,15022.0,2022
7,Batam,240.0,329.0,995.0,8136.0,16741.0,39649.0,56610.0,59750.0,77981.0,77752.0,80216.0,144521.0,562920.0,2022
9,Entikong,0.0,0.0,2.0,254.0,556.0,789.0,646.0,745.0,1360.0,1804.0,1771.0,3877.0,11804.0,2022
10,Hasanuddin,0.0,0.0,0.0,164.0,364.0,523.0,706.0,503.0,701.0,897.0,526.0,1530.0,5914.0,2022
11,Husein Sastranegara,0.0,0.0,0.0,0.0,0.0,49.0,21.0,15.0,0.0,0.0,0.0,0.0,85.0,2022
12,Jayapura,0.0,0.0,0.0,20.0,0.0,0.0,0.0,0.0,0.0,78.0,166.0,184.0,448.0,2022


In [66]:
df_download_final.to_csv(
    "data/process/hasil_download_final.csv",
    index=False
)

print("Dataset final berhasil disimpan!")

Dataset final berhasil disimpan!


In [67]:
print("JUMLAH BARIS:", len(df_download_final))
print("JUMLAH KOLOM:", len(df_download_final.columns))

print("\nDATA BERDASARKAN TAHUN:")
print(
    df_download_final["Tahun"]
    .value_counts()
    .sort_index()
)

print("\nJUMLAH DATA KOSONG:")
print(df_download_final.isnull().sum())

print("\nJUMLAH DUPLIKASI:")
print(df_download_final.duplicated().sum())

JUMLAH BARIS: 116
JUMLAH KOLOM: 15

DATA BERDASARKAN TAHUN:
Tahun
2022    28
2023    28
2024    30
2025    30
Name: count, dtype: int64

JUMLAH DATA KOSONG:
Pintu Masuk    0
Januari        1
Februari       1
Maret          1
April          1
Mei            1
Juni           1
Juli           1
Agustus        1
September      1
Oktober        1
November       1
Desember       0
Tahunan        0
Tahun          0
dtype: int64

JUMLAH DUPLIKASI:
0


In [68]:
import hashlib
import os

# Lokasi file dataset asli
file_asli = (
    "data/raw/download/"
    "Jumlah Kunjungan Wisatawan Mancanegara per bulan ke Indonesia "
    "Menurut Pintu Masuk, 2008 - sekarang, 2022.csv"
)

# Menghitung MD5
def hitung_md5(lokasi):
    md5 = hashlib.md5()

    with open(lokasi, "rb") as file:
        for bagian in iter(lambda: file.read(4096), b""):
            md5.update(bagian)

    return md5.hexdigest()

print("MD5:", hitung_md5(file_asli))
print("Ukuran file:", os.path.getsize(file_asli), "byte")

MD5: daba6b473fdf84e7d075cc32e7c22122
Ukuran file: 2706 byte


In [69]:
import hashlib
import os
import pandas as pd

# Fungsi menghitung MD5
def hitung_md5(lokasi):
    md5 = hashlib.md5()

    with open(lokasi, "rb") as file:
        for bagian in iter(lambda: file.read(4096), b""):
            md5.update(bagian)

    return md5.hexdigest()


# Menghitung ukuran dan MD5 semua dataset
daftar_metadata = []

for file in sorted(file_csv):

    lokasi = os.path.join(folder_data, file)

    metadata = {
        "nama_file": file,
        "ukuran_byte": os.path.getsize(lokasi),
        "md5": hitung_md5(lokasi)
    }

    daftar_metadata.append(metadata)


# Menampilkan hasil
df_metadata = pd.DataFrame(daftar_metadata)

display(df_metadata)

,nama_file,ukuran_byte,md5
0,Jumlah Kunjungan Wisatawan Mancanegara per bul...,2706,daba6b473fdf84e7d075cc32e7c22122
1,Jumlah Kunjungan Wisatawan Mancanegara per bul...,3065,41bd665fdc69490698b230d73c099d94
2,Jumlah Kunjungan Wisatawan Mancanegara per bul...,3112,871c2d3afa1f43fd7aa02bb116a120fb
3,Jumlah Kunjungan Wisatawan Mancanegara per bul...,3188,0a25d32a1fc15e65974727e20f22a783


In [70]:
# Membuat data provenance

from datetime import date

daftar_provenance = []

for file in sorted(file_csv):

    lokasi = os.path.join(folder_data, file)

    daftar_provenance.append({
        "nama_sumber": file,
        "url": "ISI_URL_SUMBER_ASLI",
        "tanggal_akses": "2026-11-24",
        "lisensi": "PERLU_VERIFIKASI",
        "md5": hitung_md5(lokasi)
    })

df_provenance = pd.DataFrame(daftar_provenance)

# Menyimpan provenance
df_provenance.to_csv(
    "docs/provenance.csv",
    index=False
)

display(df_provenance)

,nama_sumber,url,tanggal_akses,lisensi,md5
0,Jumlah Kunjungan Wisatawan Mancanegara per bul...,ISI_URL_SUMBER_ASLI,2026-11-24,PERLU_VERIFIKASI,daba6b473fdf84e7d075cc32e7c22122
1,Jumlah Kunjungan Wisatawan Mancanegara per bul...,ISI_URL_SUMBER_ASLI,2026-11-24,PERLU_VERIFIKASI,41bd665fdc69490698b230d73c099d94
2,Jumlah Kunjungan Wisatawan Mancanegara per bul...,ISI_URL_SUMBER_ASLI,2026-11-24,PERLU_VERIFIKASI,871c2d3afa1f43fd7aa02bb116a120fb
3,Jumlah Kunjungan Wisatawan Mancanegara per bul...,ISI_URL_SUMBER_ASLI,2026-11-24,PERLU_VERIFIKASI,0a25d32a1fc15e65974727e20f22a783


In [71]:
# Membuat provenance untuk 4 dataset BPS

daftar_provenance = []

for file in sorted(file_csv):

    lokasi = os.path.join(folder_data, file)

    daftar_provenance.append({
        "nama_sumber": file,
        "url": "https://www.bps.go.id/id/statistics-table/2/MTE1MCMy/kunjungan-wisman-ke-indonesia--september-2026.html",
        "tanggal_akses": "2026-11-24",
        "lisensi": "Belum dinyatakan pada halaman sumber",
        "md5": hitung_md5(lokasi)
    })

df_provenance = pd.DataFrame(daftar_provenance)

# Menyimpan provenance
df_provenance.to_csv(
    "docs/provenance.csv",
    index=False
)

display(df_provenance)

,nama_sumber,url,tanggal_akses,lisensi,md5
0,Jumlah Kunjungan Wisatawan Mancanegara per bul...,https://www.bps.go.id/id/statistics-table/2/MT...,2026-11-24,Belum dinyatakan pada halaman sumber,daba6b473fdf84e7d075cc32e7c22122
1,Jumlah Kunjungan Wisatawan Mancanegara per bul...,https://www.bps.go.id/id/statistics-table/2/MT...,2026-11-24,Belum dinyatakan pada halaman sumber,41bd665fdc69490698b230d73c099d94
2,Jumlah Kunjungan Wisatawan Mancanegara per bul...,https://www.bps.go.id/id/statistics-table/2/MT...,2026-11-24,Belum dinyatakan pada halaman sumber,871c2d3afa1f43fd7aa02bb116a120fb
3,Jumlah Kunjungan Wisatawan Mancanegara per bul...,https://www.bps.go.id/id/statistics-table/2/MT...,2026-11-24,Belum dinyatakan pada halaman sumber,0a25d32a1fc15e65974727e20f22a783


In [72]:
df_provenance = pd.read_csv("docs/provenance.csv")

print("Jumlah sumber:", len(df_provenance))

display(df_provenance)

Jumlah sumber: 4


,nama_sumber,url,tanggal_akses,lisensi,md5
0,Jumlah Kunjungan Wisatawan Mancanegara per bul...,https://www.bps.go.id/id/statistics-table/2/MT...,2026-11-24,Belum dinyatakan pada halaman sumber,daba6b473fdf84e7d075cc32e7c22122
1,Jumlah Kunjungan Wisatawan Mancanegara per bul...,https://www.bps.go.id/id/statistics-table/2/MT...,2026-11-24,Belum dinyatakan pada halaman sumber,41bd665fdc69490698b230d73c099d94
2,Jumlah Kunjungan Wisatawan Mancanegara per bul...,https://www.bps.go.id/id/statistics-table/2/MT...,2026-11-24,Belum dinyatakan pada halaman sumber,871c2d3afa1f43fd7aa02bb116a120fb
3,Jumlah Kunjungan Wisatawan Mancanegara per bul...,https://www.bps.go.id/id/statistics-table/2/MT...,2026-11-24,Belum dinyatakan pada halaman sumber,0a25d32a1fc15e65974727e20f22a783


In [73]:
# Mencari file HTML di folder scraping

folder_scraping = "data/raw/scraping"

for file in os.listdir(folder_scraping):
    print(file)

day_bali_tour.html


In [74]:
# Menghitung metadata file HTML

for file in os.listdir(folder_scraping):

    lokasi = os.path.join(folder_scraping, file)

    if file.endswith(".html"):
        print("Nama file:", file)
        print("Ukuran:", os.path.getsize(lokasi), "byte")
        print("MD5:", hitung_md5(lokasi))

Nama file: day_bali_tour.html
Ukuran: 391767 byte
MD5: a2d4750647a51c5252cb8b8bf0d28996


In [75]:
# Membaca provenance yang sudah ada
df_provenance = pd.read_csv("docs/provenance.csv")

# Membuat data sumber scraping
data_scraping = {
    "nama_sumber": "day_bali_tour.html",
    "url": "https://www.daybalitour.com/best-bali-day-tour-packages/",
    "tanggal_akses": "2026-11-24",
    "lisensi": "Belum dinyatakan pada halaman sumber",
    "md5": "a2d4750647a51c5252cb8b8bf0d28996"
}

# Menambahkan sumber scraping ke metadata
df_provenance = pd.concat(
    [df_provenance, pd.DataFrame([data_scraping])],
    ignore_index=True
)

# Menyimpan kembali provenance
df_provenance.to_csv("docs/provenance.csv", index=False)

# Menampilkan hasil
display(df_provenance)

,nama_sumber,url,tanggal_akses,lisensi,md5
0,Jumlah Kunjungan Wisatawan Mancanegara per bul...,https://www.bps.go.id/id/statistics-table/2/MT...,2026-11-24,Belum dinyatakan pada halaman sumber,daba6b473fdf84e7d075cc32e7c22122
1,Jumlah Kunjungan Wisatawan Mancanegara per bul...,https://www.bps.go.id/id/statistics-table/2/MT...,2026-11-24,Belum dinyatakan pada halaman sumber,41bd665fdc69490698b230d73c099d94
2,Jumlah Kunjungan Wisatawan Mancanegara per bul...,https://www.bps.go.id/id/statistics-table/2/MT...,2026-11-24,Belum dinyatakan pada halaman sumber,871c2d3afa1f43fd7aa02bb116a120fb
3,Jumlah Kunjungan Wisatawan Mancanegara per bul...,https://www.bps.go.id/id/statistics-table/2/MT...,2026-11-24,Belum dinyatakan pada halaman sumber,0a25d32a1fc15e65974727e20f22a783
4,day_bali_tour.html,https://www.daybalitour.com/best-bali-day-tour...,2026-11-24,Belum dinyatakan pada halaman sumber,a2d4750647a51c5252cb8b8bf0d28996


In [76]:
# Melihat hasil scraping
display(df_scraping.head())

,nama_paket,harga,informasi
0,Bali Day Tours Highlights,None,Bali Day Tours Highlights
1,Bali Waterfall and Best Ubud Tour Packages,IDR 580.000,Bali Waterfall and Best Ubud Tour Packages Fro...
2,From IDR 580.000,IDR 580.000,From IDR 580.000 What’s Included: Private Hote...
3,Bali Best Nusa Penida Island Tour Packages,IDR 760.000,Bali Best Nusa Penida Island Tour Packages Fro...
4,From IDR 760.000,IDR 760.000,From IDR 760.000 What’s Included: Private Hote...


In [80]:
# 1. Menghapus baris tanpa harga
df_scraping_bersih = df_scraping.dropna(
    subset=["harga"]
).copy()

# 2. Menghapus baris yang bukan nama paket wisata
df_scraping_bersih = df_scraping_bersih[
    ~df_scraping_bersih["nama_paket"].str.startswith(
        ("From", "IDR"), na=False
    )
].copy()

# 3. Menghapus data duplikat
df_scraping_bersih = df_scraping_bersih.drop_duplicates()

# 4. Mengatur ulang indeks
df_scraping_bersih = df_scraping_bersih.reset_index(drop=True)

# 5. Menampilkan hasil
display(df_scraping_bersih)

# 6. Menghitung jumlah data
print("Jumlah data:", len(df_scraping_bersih))

,nama_paket,harga,informasi
0,Bali Waterfall and Best Ubud Tour Packages,IDR 580.000,Bali Waterfall and Best Ubud Tour Packages Fro...
1,Bali Best Nusa Penida Island Tour Packages,IDR 760.000,Bali Best Nusa Penida Island Tour Packages Fro...
2,Bali Lempuyang Gate of Heaven Tour Package,IDR 745.000,Bali Lempuyang Gate of Heaven Tour Package Fro...
3,Bali Ulun Danu and Tanah Lot Tour Package,IDR 790.000,Bali Ulun Danu and Tanah Lot Tour Package From...
4,Bali Tanjung Benoa & Uluwatu Tour Package,IDR 870.000,Bali Tanjung Benoa & Uluwatu Tour Package From...
5,Bali Kintamani Volcano View Tour Package,IDR 715.000,Bali Kintamani Volcano View Tour Package From ...
6,Bali Driver with Toyota Innova,IDR 1.125.000,Bali Driver with Toyota Innova IDR 1.125.000/C...
7,Bali Driver with Toyota Hiace,IDR 1.275.000,Bali Driver with Toyota Hiace IDR 1.275.000/Ca...


Jumlah data: 8


In [81]:
# Menyimpan hasil scraping yang sudah dibersihkan
df_scraping_bersih.to_csv(
    "data/process/hasil_scraping_daybalitour.csv",
    index=False
)

print("Data scraping berhasil disimpan!")

Data scraping berhasil disimpan!


In [82]:
import os

file_scraping = "data/process/hasil_scraping_daybalitour.csv"

print("File tersedia:", os.path.exists(file_scraping))
print("Ukuran file:", os.path.getsize(file_scraping), "byte")

File tersedia: True
Ukuran file: 2217 byte


In [83]:
# Memeriksa informasi dataset
print("Informasi dataset:")
df_scraping_bersih.info()

# Memeriksa nilai kosong
print("\nJumlah nilai kosong:")
print(df_scraping_bersih.isnull().sum())

# Memeriksa data duplikat
print("\nJumlah baris duplikat:")
print(df_scraping_bersih.duplicated().sum())

Informasi dataset:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8 entries, 0 to 7
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   nama_paket  8 non-null      object
 1   harga       8 non-null      object
 2   informasi   8 non-null      object
dtypes: object(3)
memory usage: 324.0+ bytes

Jumlah nilai kosong:
nama_paket    0
harga         0
informasi     0
dtype: int64

Jumlah baris duplikat:
0


In [84]:
# Menampilkan nama paket dan harga
display(
    df_scraping_bersih[["nama_paket", "harga"]]
)

,nama_paket,harga
0,Bali Waterfall and Best Ubud Tour Packages,IDR 580.000
1,Bali Best Nusa Penida Island Tour Packages,IDR 760.000
2,Bali Lempuyang Gate of Heaven Tour Package,IDR 745.000
3,Bali Ulun Danu and Tanah Lot Tour Package,IDR 790.000
4,Bali Tanjung Benoa & Uluwatu Tour Package,IDR 870.000
5,Bali Kintamani Volcano View Tour Package,IDR 715.000
6,Bali Driver with Toyota Innova,IDR 1.125.000
7,Bali Driver with Toyota Hiace,IDR 1.275.000


In [85]:
# Menghapus tulisan IDR dan titik pemisah ribuan
df_scraping_bersih["harga"] = (
    df_scraping_bersih["harga"]
    .str.replace("IDR", "", regex=False)
    .str.replace(".", "", regex=False)
    .str.strip()
    .astype(int)
)

# Memeriksa hasil konversi
display(df_scraping_bersih[["nama_paket", "harga"]])

# Memeriksa tipe data
print(df_scraping_bersih.dtypes)

,nama_paket,harga
0,Bali Waterfall and Best Ubud Tour Packages,580000
1,Bali Best Nusa Penida Island Tour Packages,760000
2,Bali Lempuyang Gate of Heaven Tour Package,745000
3,Bali Ulun Danu and Tanah Lot Tour Package,790000
4,Bali Tanjung Benoa & Uluwatu Tour Package,870000
5,Bali Kintamani Volcano View Tour Package,715000
6,Bali Driver with Toyota Innova,1125000
7,Bali Driver with Toyota Hiace,1275000


nama_paket    object
harga          int64
informasi     object
dtype: object


In [86]:
# Menghitung statistik harga
print("Harga termurah: Rp", df_scraping_bersih["harga"].min())
print("Harga termahal: Rp", df_scraping_bersih["harga"].max())
print("Rata-rata harga: Rp", round(df_scraping_bersih["harga"].mean()))

# Statistik lengkap
display(df_scraping_bersih["harga"].describe())

Harga termurah: Rp 580000
Harga termahal: Rp 1275000
Rata-rata harga: Rp 857500


,harga
count,8.000000e+00
mean,8.575000e+05
std,2.299223e+05
min,5.800000e+05
25%,7.375000e+05
50%,7.750000e+05
75%,9.337500e+05
max,1.275000e+06


In [87]:
df_scraping_bersih.to_csv(
    "data/process/hasil_scraping_daybalitour.csv",
    index=False
)

print("Dataset terbaru berhasil disimpan!")

Dataset terbaru berhasil disimpan!
